In [1]:
import pandas as pd

In [2]:
import sys
sys.path.append('../../libs')

%load_ext autoreload
%autoreload 2

import tools_comparison
import categorical

# import ios
# import vis
# from tools_comparison import ToolsComparison
# import constants

Total combinations of features: 1023


# Data

## Ana

In [10]:
df_two_ch = pd.read_csv("_gt_sampling/DDBB_final_new_filtered_GT_multiple_characters.csv", delimiter="|").set_index('givenname')
df_two_ch.loc[:, 'single_char'] = False

df_one_ch = pd.read_csv("_gt_sampling/DDBB_final_new_filtered_GT_unique_characters.csv", delimiter="|").set_index('givenname').rename(columns={'Same_gender':'gender_nr'})
df_one_ch.loc[:, 'single_char'] = True

df_ana = pd.concat([df_two_ch, df_one_ch], ignore_index=False)
df_ana.rename(columns={'Gender_w':'gender_w', 'Gender_r':'gender_r', 'Gender_n':'gender_n', 
                       'Same_gender_n_w':'same_gender_nw',
                       'Same_gender_all':'same_gender_nrw',
                       'Same_gender_n_r':'same_gender_nr',
                       'Same_gender_w_r':'same_gender_rw',
                       'Most_frequent_gender':'most_frequent'
                       }, inplace=True)

## Lisette

In [24]:
cols = ['givenname', 'ngender_n', 'ngender_r', 'gender_n', 'gender_w', 'gender_nw', 'single_char']
df_gt = pd.read_csv('../../../results/v3/gender/predictions/CNKI_uniqueChineseAllGivenNames_gender.csv').query("gt==True")

df_gt = df_gt[cols].set_index('givenname')

df_gt.loc[:,'gender_r'] = df_gt.ngender_r.apply(lambda v: categorical.UNKNOWN if tools_comparison.pd.isna(v) else categorical.FEMALE if v>0.5 else categorical.MALE if v<0.5 else categorical.NEUTRAL)

df_gt.loc[:,'gender_nrw'] = df_gt.apply(lambda row: 
                                                row.gender_n if row.gender_n == row.gender_r and row.gender_w == categorical.UNKNOWN and row.single_char else 
                                                row.gender_n if row.gender_n == row.gender_w and row.gender_n == row.gender_r else 
                                                categorical.MOSTLY_FEMALE if row.gender_n == row.gender_w and row.gender_n != row.gender_r and row.gender_n == categorical.FEMALE else 
                                                categorical.MOSTLY_FEMALE if row.gender_n == row.gender_r and row.gender_n != row.gender_w and row.gender_n == categorical.FEMALE else 
                                                categorical.MOSTLY_FEMALE if row.gender_r == row.gender_w and row.gender_r != row.gender_n and row.gender_n == categorical.FEMALE else
                                                categorical.MOSTLY_MALE if row.gender_n == row.gender_w and row.gender_n != row.gender_r and row.gender_n == categorical.MALE else 
                                                categorical.MOSTLY_MALE if row.gender_n == row.gender_r and row.gender_n != row.gender_w and row.gender_n == categorical.MALE else 
                                                categorical.MOSTLY_MALE if row.gender_r == row.gender_w and row.gender_r != row.gender_n and row.gender_n == categorical.MALE else
                                                categorical.NEUTRAL, axis=1)


df_gt.loc[:, 'same_gender_nrw'] = df_gt.gender_nrw.apply(lambda v: v != categorical.NEUTRAL)
for tool1, tool2 in [('n','r'), ('n','w'), ('r', 'w')]:
    df_gt.loc[:, f'same_gender_{tool1}{tool2}'] = df_gt.apply(lambda row: row[f'gender_{tool1}'] == row[f'gender_{tool2}'], axis=1)



In [25]:
df_two_ch.shape, df_one_ch.shape, df_ana.shape, df_gt.shape

((116, 9), (50, 4), (166, 10), (166, 12))

In [26]:
df_ana.head(2)



,gender_w,gender_r,gender_n,same_gender_nrw,same_gender_nr,same_gender_nw,same_gender_rw,most_frequent,single_char,gender_nr
givenname,,,,,,,,,,
一葳,female,male,male,False,True,False,False,male,False,NaN
丙辰,female,male,male,False,True,False,False,male,False,NaN


In [27]:
df_gt.head(2)

,ngender_n,ngender_r,gender_n,gender_w,gender_nw,single_char,gender_r,gender_nrw,same_gender_nrw,same_gender_nr,same_gender_nw,same_gender_rw
givenname,,,,,,,,,,,,
东,0.451906,0.15675,male,unknown,male,True,male,male,True,True,False,False
习,0.460552,0.34495,male,unknown,male,True,male,male,True,True,False,False


### Comparing raw tool values

In [41]:
ind_cols

['gender_n']

In [47]:
tools_cols = set(df_ana.columns.tolist()).intersection(set(df_gt.columns.tolist()))
results_single_tools = {}
for col in tools_cols:

    ind_cols = []
    if col != 'single_char':
        ind_cols = [f"gender_{v}" for v in col.split('_')[-1]]
        if len(ind_cols) == 1:
            ind_cols = []

    _df = df_ana[[col]+ind_cols].join(df_gt[[col]+ind_cols], lsuffix='_A', rsuffix='_L')
    _df.loc[:, f'AL_{col}'] = _df.apply(lambda row: '-' if ~pd.notna(row[f"{col}_A"]) and row[f"{col}_L"] == categorical.UNKNOWN else row[f"{col}_A"] == row[f"{col}_L"], axis=1)
    results_single_tools[col] = _df

/tmp/ipykernel_2149774/1101942121.py:12: DeprecationWarning: Bitwise inversion '~' on bool is deprecated. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  _df.loc[:, f'AL_{col}'] = _df.apply(lambda row: '-' if ~pd.notna(row[f"{col}_A"]) and row[f"{col}_L"] == categorical.UNKNOWN else row[f"{col}_A"] == row[f"{col}_L"], axis=1)
/tmp/ipykernel_2149774/1101942121.py:12: DeprecationWarning: Bitwise inversion '~' on bool is deprecated. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  _df.loc[:, f'AL_{col}'] = _df.apply(lambda row: '-' if ~pd.notna(row[f"{col}_A"]) and row[f"{col}_L"] == categorical.UNKNOWN else 

In [48]:
for col, _df in results_single_tools.items():
    tmp = _df.query(f"AL_{col}==False")

    if tmp.shape[0] > 0:
        print(f"=== {col} ===")
        print(tmp.shape)
        print(tmp)
        print('')

=== gender_n ===
(57, 3)
          gender_n_A gender_n_L AL_gender_n
givenname                                  
东业            female       male       False
东霞              male     female       False
丹雯              male     female       False
丽姣              male     female       False
丽存              male     female       False
亚天            female       male       False
亚平            female       male       False
佳玉              male     female       False
冬艳              male     female       False
华莲              male     female       False
博华            female       male       False
国钦            female       male       False
大卉              male     female       False
奕初            female       male       False
婧翔            female       male       False
婷贤            female       male       False
宇丹              male     female       False
帮美              male     female       False
建兰              male     female       False
彦喆            female       male       False
彩凤     

In [50]:
df_ana.loc['业俊']

gender_w            male
gender_r            male
gender_n            male
same_gender_nrw     True
same_gender_nr     False
same_gender_nw     False
same_gender_rw     False
most_frequent       male
single_char        False
gender_nr            NaN
Name: 业俊, dtype: object